In [3]:
# Brightdata datacenter proxy — replace with your actual credentials
BD_HOST = "brd.superproxy.io"
BD_PORT = 33335
BD_USER = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
BD_PASS = "ymg5cg3a1z33"

def get_proxy_url() -> str:
    return f"http://{BD_USER}:{BD_PASS}@{BD_HOST}:{BD_PORT}"

def get_httpx_proxies() -> dict:
    """For httpx AsyncClient / Client"""
    url = get_proxy_url()
    return {"http://": url, "https://": url}

def get_playwright_proxy() -> dict:
    """For Playwright browser context"""
    return {
        "server": f"http://{BD_HOST}:{BD_PORT}",
        "username": BD_USER,
        "password": BD_PASS,
    }
import nest_asyncio
nest_asyncio.apply()

In [2]:
import asyncio
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import httpx
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

# ── proxy helpers (your existing functions) ───────────────────────────────────

BASE       = "https://www.arrow.com"
SEARCH_API = f"{BASE}/productsearch/productlinesearchresultajax"
PER_PAGE   = 50
OUT_FILE   = Path("discovery.json")

HEADERS = {
    "user-agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
    ),
    "accept-language": "en-US,en;q=0.9",
}

In [4]:
def load_discovery() -> list[dict]:
    raw  = json.loads(DISCOVERY.read_text())
    seen, flat = set(), []
    for cat in raw["categories"].values():
        for p in cat["products"]:
            pn = p.get("partNumber")
            if pn and pn not in seen:
                seen.add(pn)
                flat.append(p)
    print(f"[load] {len(flat)} unique products from discovery.json")
    return flat

def load_results() -> dict:
    if OUT_FILE.exists():
        data = json.loads(OUT_FILE.read_text())
        done   = sum(1 for p in data["products"].values() if p["source"] != "failed")
        failed = sum(1 for p in data["products"].values() if p["source"] == "failed")
        print(f"[load] resuming — {done} done, {failed} failed")
        return data
    return {"meta": {}, "products": {}}

def save_results(out: dict):
    done   = sum(1 for p in out["products"].values() if p["source"] != "failed")
    failed = sum(1 for p in out["products"].values() if p["source"] == "failed")
    out["meta"] = {
        "total":      len(out["products"]),
        "done":       done,
        "failed":     failed,
        "scraped_at": datetime.now(timezone.utc).isoformat(),
    }
    OUT_FILE.write_text(json.dumps(out, indent=2))

In [5]:
@asynccontextmanager
async def fresh_client(cookies: dict = None):
    async with httpx.AsyncClient(
        proxies=get_httpx_proxies(),
        headers=HEADERS,
        cookies=cookies or {},
        timeout=30,
        follow_redirects=True,
    ) as client:
        yield client


async def get_session_cookies(seed_url: str) -> dict:
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-http2"],
            proxy=get_playwright_proxy(),
        )
        ctx  = await browser.new_context(proxy=get_playwright_proxy())
        page = await ctx.new_page()

        await page.route("**/*", lambda route, req: (
            route.abort() if req.resource_type in ("image", "font", "media", "stylesheet")
            else route.continue_()
        ))

        await page.goto(seed_url, wait_until="domcontentloaded", timeout=60_000)
        await page.wait_for_timeout(2_000)

        raw = await ctx.cookies()
        await browser.close()

    cookies = {c["name"]: c["value"] for c in raw}
    print(f"[bootstrap] got {len(cookies)} cookies")
    return cookies

In [6]:
async def get_pdp(
    url: str,
    product_detail_str: str,
    cookies: dict,
    retries: int = RETRIES,
) -> dict:
    for attempt in range(retries):
        try:
            async with fresh_client(cookies) as client:
                r = await client.post(
                    PRICING_API,
                    json={"productDetails": [product_detail_str]},
                    headers={"content-type": "application/json", "referer": f"{BASE}/"},
                )
            if r.status_code == 200:
                data = r.json()
                results = data.get("results") or data.get("data") or []
                if results:
                    return {"source": "api", "data": data}
            print(f"  [api] attempt {attempt+1} got {r.status_code}, retrying…")
        except Exception as e:
            print(f"  [api] attempt {attempt+1} error: {e}, retrying…")
        await asyncio.sleep(1)

    print(f"  [html] falling back → {url}")
    for attempt in range(retries):
        try:
            async with fresh_client(cookies) as client:
                r = await client.get(url)

            if r.status_code != 200:
                print(f"  [html] attempt {attempt+1} got {r.status_code}")
                await asyncio.sleep(1)
                continue

            soup    = BeautifulSoup(r.text, "html.parser")
            ld_json = {}
            script  = soup.find("script", {"type": "application/ld+json"})
            if script and script.string:
                try:
                    ld_json = json.loads(script.string)
                except json.JSONDecodeError:
                    pass

            specs = {}
            for row in soup.select("tr.attribute-row, table.attributes-table tr"):
                cells = row.select("td, th")
                if len(cells) >= 2:
                    specs[cells[0].get_text(strip=True)] = cells[1].get_text(strip=True)

            return {"source": "html", "ld_json": ld_json, "specs": specs}

        except Exception as e:
            print(f"  [html] attempt {attempt+1} error: {e}")
            await asyncio.sleep(1)

    return {"source": "failed", "url": url}

NameError: name 'RETRIES' is not defined

In [ ]:
async def run_pdp():
    products = load_discovery()
    out      = load_results()
    cookies  = await get_session_cookies(f"{BASE}/en/categories")

    # only queue products not yet successfully scraped
    todo = [
        p for p in products
        if p["partNumber"] not in out["products"]
    ]
    print(f"[pdp] {len(todo)} products to scrape ({len(products) - len(todo)} skipped)")

    for i, p in enumerate(todo, 1):
        pn         = p["partNumber"]
        url        = p["pdpUrl"]
        detail_str = f"{p['arrowId']}|{p['manufacturer']}|{pn}|{p.get('eccn','EAR99')}|null"

        result = await get_pdp(url, detail_str, cookies)
        out["products"][pn] = {
            "source":       result["source"],
            "partNumber":   pn,
            "manufacturer": p["manufacturer"],
            "pdpUrl":       url,
            "data":         result.get("data") or result,
        }

        print(f"  [{i}/{len(todo)}] {pn} → {result['source']}")

        # save every 50 products so progress is never lost
        if i % 50 == 0:
            save_results(out)

    save_results(out)
    print(f"\n[done] {out['meta']['done']} ok, {out['meta']['failed']} failed")

await run_pdp()